# MIT 805 Group 19 - Part 1

## Imports

In [ ]:
import numpy as np
import pandas as pd
import csv
import math
import os
import time
import datetime as date
from pathlib import Path
import requests
from pyspark.sql import SparkSession

## Data Collection

In [ ]:
# NYC TLC Trip Record Data

# Data directory
data_dir = Path("data")
raw_catalog_dir = data_dir / "raw_data.csv" # if needed to document raw data characteristics
working_dir = data_dir / "working" 
schema_ref_dir = data_dir / "schema reference" 

for d in (data_dir, working_dir, schema_ref_dir):
    d.mkdir(parents=True, exist_ok=True)

# Data to be pulled
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/{service}_tripdata_{yyyy}-{mm:02d}.parquet"
services = ["yellow", "green", "fhv", "fhvhv"]
year_start, month_start = 2015, 1
year_end, month_end = 2026, 5

primary_service = "yellow" 
working_data_target = 12.5 # size of working dataset
working_data_cap = 15 # working dataset should not exceed this size

headers = {"User-Agent": "MIT805-project/1.0 (student data-collection script)"}

def month_range(y0, m0, y1, m1):
    y, m = y0, m0
    while (y, m) <= (y1, m1):
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)

def file_url(service, year, month):
    return base_url.format(service=service, yyyy=year, mm=month)